# Decision trees and random forests from scratch (numpy only)

This notebook goes with `notes.md` §5 (decision trees) and §6.1-§6.2 (bagging
and random forests). It builds a binary classification tree with entropy
(information-gain) splits, shows how train and test accuracy change with
depth, builds a random forest on top of the tree, and compares both with
sklearn. The last two sections measure the quantities in the bagging variance
formula $\operatorname{Var}(\bar f) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$:
the tree-to-tree correlation $\rho$ and the single-tree variance $\sigma^2$,
for bagging and for random forests, across values of `max_features`.

All randomness comes from fixed seeds, and the notebook runs in well under a
minute on a CPU.

In [1]:
import numpy as np

np.random.seed(42)

## 1. Entropy and information gain (notes §5.1-§5.2)

`entropy` computes $H(Y) = -\sum_c p_c \log_2 p_c$ in bits. As a check, the
cell reproduces two numbers from the notes: a 90/10 label has entropy 0.469
bits, and splitting a 5/5 parent into children (4+, 1-) and (1+, 4-) gives an
information gain of 0.278 bits.

In [2]:
def entropy(y):
    # H(y) = -sum_c p_c log2 p_c, in bits.
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return float(-np.sum(p * np.log2(p)))


h_90_10 = entropy(np.array([1] * 9 + [0]))
parent = np.array([1] * 5 + [0] * 5)
left, right = np.array([1] * 4 + [0]), np.array([1] + [0] * 4)
ig = entropy(parent) - 0.5 * entropy(left) - 0.5 * entropy(right)
print(f"H(90/10) = {h_90_10:.3f} bits")
print(f"IG of the (4+,1-) / (1+,4-) split = {ig:.3f} bits")
assert round(h_90_10, 3) == 0.469 and round(ig, 3) == 0.278
print("PASS: matches the numbers in notes §5.1-§5.2.")

H(90/10) = 0.469 bits
IG of the (4+,1-) / (1+,4-) split = 0.278 bits
PASS: matches the numbers in notes §5.1-§5.2.


## 2. Data: XOR of two signs, plus noise features

Two informative features are drawn uniformly from $[-2, 2]$ and the label is 1
when they have the same sign (an XOR pattern). Three pure-noise features are
appended, and 5% of labels are flipped. No single feature predicts the label
on its own (each half of either axis is about 50/50), so no linear model can
fit this, and a tree has to find the interaction through a sequence of
splits. The first 800 points are the training set and the last 400 the test
set; the majority-class baseline is near 50%.

In [3]:
def make_data(n=1200, d_noise=3):
    X_inf = np.random.uniform(-2, 2, size=(n, 2))
    y = ((X_inf[:, 0] * X_inf[:, 1]) > 0).astype(int)   # XOR of the signs
    flip = np.random.rand(n) < 0.05                       # 5% label noise
    y[flip] = 1 - y[flip]
    X_noise = np.random.randn(n, d_noise)
    return np.hstack([X_inf, X_noise]), y


X, y = make_data()
n_train = 800
Xtr, ytr = X[:n_train], y[:n_train]
Xte, yte = X[n_train:], y[n_train:]
print(f"data: {Xtr.shape[0]} train / {Xte.shape[0]} test, "
      f"{X.shape[1]} features (2 informative, 3 noise)")
print(f"majority-class baseline test acc = {max(np.mean(yte), 1 - np.mean(yte)):.4f}")

data: 800 train / 400 test, 5 features (2 informative, 3 noise)
majority-class baseline test acc = 0.5125


## 3. A decision tree, and how depth trades bias for variance (notes §5.2-§5.3)

The tree is built greedily: at each node, try every feature and every
threshold (midpoints between consecutive sorted unique values), keep the split
with the largest information gain, and recurse until `max_depth`, a pure node,
or too few samples. To score all thresholds of a feature at once, the code
sorts that feature and uses cumulative counts of class 1 to get each
candidate's left and right entropies, which gives the same splits as testing
each threshold separately, only faster. `max_features` (used by the random
forest later) restricts each split to a random subset of features.

The sweep over depth should show train accuracy rising to 1.0 as depth grows,
while test accuracy peaks at a moderate depth and then falls: a deep tree
fits the flipped labels too (notes §5.3).

In [4]:
def binary_entropy(p):
    # Entropy in bits of a Bernoulli(p), elementwise, with 0 log 0 = 0.
    p = np.asarray(p, dtype=float)
    out = np.zeros_like(p)
    m = (p > 0) & (p < 1)
    q = p[m]
    out[m] = -(q * np.log2(q) + (1 - q) * np.log2(1 - q))
    return out


class Node:
    __slots__ = ("feature", "threshold", "left", "right", "prediction", "proba")

    def __init__(self, feature=None, threshold=None, left=None, right=None,
                 prediction=None, proba=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction   # set on leaves only
        self.proba = proba             # fraction of class 1 at the leaf


class DecisionTree:
    # Binary classification tree with entropy (information-gain) splits.

    def __init__(self, max_depth=5, min_samples_split=2, max_features=None,
                 rng=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features  # None = consider all features
        self.rng = rng if rng is not None else np.random.default_rng(0)
        self.root = None

    def fit(self, X, y):
        self.root = self._build(X, y, depth=0)
        return self

    def _best_split(self, X, y):
        # Return (feature, threshold, info_gain) of the best split, or None.
        n, d = X.shape
        parent_H = entropy(y)
        if self.max_features is None:
            features = np.arange(d)
        else:
            features = self.rng.choice(d, size=min(self.max_features, d),
                                       replace=False)
        n_l = np.arange(1, n)          # left-child size if we split after row i
        n_r = n - n_l
        total_ones = y.sum()
        best = None                    # (gain, feature, threshold)
        for f in features:
            order = np.argsort(X[:, f], kind="stable")
            xs, ys = X[order, f], y[order]
            valid = xs[1:] > xs[:-1]   # only split between distinct values
            if not valid.any():
                continue
            ones_l = np.cumsum(ys)[:-1]
            ones_r = total_ones - ones_l
            gain = (parent_H
                    - (n_l / n) * binary_entropy(ones_l / n_l)
                    - (n_r / n) * binary_entropy(ones_r / n_r))
            gain[~valid] = -np.inf
            i = int(np.argmax(gain))
            if best is None or gain[i] > best[0]:
                best = (gain[i], f, (xs[i] + xs[i + 1]) / 2.0)
        if best is None or best[0] <= 1e-12:
            return None
        return best[1], best[2], best[0]

    def _build(self, X, y, depth):
        if (depth >= self.max_depth
                or len(y) < self.min_samples_split
                or len(np.unique(y)) == 1):
            return self._leaf(y)
        split = self._best_split(X, y)
        if split is None:
            return self._leaf(y)
        f, t, _gain = split
        mask = X[:, f] <= t
        left = self._build(X[mask], y[mask], depth + 1)
        right = self._build(X[~mask], y[~mask], depth + 1)
        return Node(feature=f, threshold=t, left=left, right=right)

    @staticmethod
    def _leaf(y):
        p1 = float(np.mean(y)) if len(y) else 0.0
        return Node(prediction=int(p1 >= 0.5), proba=p1)

    def _leaf_of(self, x):
        node = self.root
        while node.prediction is None:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node

    def predict(self, X):
        return np.array([self._leaf_of(x).prediction for x in X])

    def predict_proba(self, X):
        return np.array([self._leaf_of(x).proba for x in X])

    def depth(self):
        def _d(node):
            if node.prediction is not None:
                return 0
            return 1 + max(_d(node.left), _d(node.right))
        return _d(self.root)


for max_depth in (2, 4, 8, 20):
    tree = DecisionTree(max_depth=max_depth).fit(Xtr, ytr)
    acc_tr = float(np.mean(tree.predict(Xtr) == ytr))
    acc_te = float(np.mean(tree.predict(Xte) == yte))
    print(f"  max_depth={max_depth:2d}: train acc = {acc_tr:.4f}, "
          f"test acc = {acc_te:.4f}, actual depth = {tree.depth()}")

  max_depth= 2: train acc = 0.6412, test acc = 0.6075, actual depth = 2
  max_depth= 4: train acc = 0.9513, test acc = 0.9525, actual depth = 4
  max_depth= 8: train acc = 0.9775, test acc = 0.9225, actual depth = 8
  max_depth=20: train acc = 1.0000, test acc = 0.9075, actual depth = 16


## 4. Check the tree against sklearn

sklearn's `DecisionTreeClassifier` with `criterion="entropy"` and the same depth
limit uses the same greedy algorithm, so its test accuracy should be within 5
percentage points of ours (small differences come from tie-breaking between
equally good splits).

In [5]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTree(max_depth=8).fit(Xtr, ytr)
acc_tree = float(np.mean(tree.predict(Xte) == yte))
sk_tree = DecisionTreeClassifier(max_depth=8, criterion="entropy",
                                 random_state=42).fit(Xtr, ytr)
sk_tree_acc = float(sk_tree.score(Xte, yte))
print(f"sklearn DecisionTree (depth 8, entropy) test acc = {sk_tree_acc:.4f} "
      f"(ours: {acc_tree:.4f})")
assert abs(sk_tree_acc - acc_tree) <= 0.05, "tree accuracy far from sklearn"
print("PASS: within 5% of sklearn's tree.")

sklearn DecisionTree (depth 8, entropy) test acc = 0.9075 (ours: 0.9225)
PASS: within 5% of sklearn's tree.


## 5. A random forest (notes §6.1-§6.2)

Each of 40 trees is trained on a bootstrap sample (800 draws with replacement
from the 800 training points) and, at every split, considers only 3 of the 5
features chosen at random. The forest predicts by majority vote. Expect test
accuracy at least as good as the single depth-8 tree from section 4, because
averaging reduces variance.

In [6]:
class RandomForest:
    # Bagging (bootstrap rows) + random feature subset at every split.

    def __init__(self, n_trees=25, max_depth=6, max_features="sqrt", seed=0):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.max_features = max_features
        self.seed = seed
        self.trees = []

    def fit(self, X, y):
        n, d = X.shape
        mf = max(1, int(np.sqrt(d))) if self.max_features == "sqrt" else self.max_features
        rng = np.random.default_rng(self.seed)
        self.trees = []
        for _ in range(self.n_trees):
            idx = rng.integers(0, n, size=n)          # bootstrap sample
            tree = DecisionTree(max_depth=self.max_depth, max_features=mf, rng=rng)
            self.trees.append(tree.fit(X[idx], y[idx]))
        return self

    def predict(self, X):
        votes = np.stack([t.predict(X) for t in self.trees])  # (n_trees, n)
        return (votes.mean(axis=0) >= 0.5).astype(int)


forest = RandomForest(n_trees=40, max_depth=8, max_features=3, seed=42).fit(Xtr, ytr)
acc_rf_tr = float(np.mean(forest.predict(Xtr) == ytr))
acc_rf = float(np.mean(forest.predict(Xte) == yte))
print(f"40 trees, depth 8, 3 of {X.shape[1]} features per split")
print(f"train acc = {acc_rf_tr:.4f}, test acc = {acc_rf:.4f}")
print(f"single depth-8 tree test acc = {acc_tree:.4f}; "
      f"forest minus tree = {acc_rf - acc_tree:+.4f}")

40 trees, depth 8, 3 of 5 features per split
train acc = 0.9738, test acc = 0.9525
single depth-8 tree test acc = 0.9225; forest minus tree = +0.0300


## 6. Check the forest against sklearn

sklearn's `RandomForestClassifier` with the same number of trees, depth,
`max_features` and criterion should reach a test accuracy within 5 percentage
points of ours. The random draws differ, so the numbers will not be identical.

In [7]:
from sklearn.ensemble import RandomForestClassifier

sk_rf = RandomForestClassifier(n_estimators=40, max_depth=8, max_features=3,
                               criterion="entropy", random_state=42).fit(Xtr, ytr)
sk_rf_acc = float(sk_rf.score(Xte, yte))
print(f"sklearn RandomForest test acc = {sk_rf_acc:.4f} (ours: {acc_rf:.4f})")
assert abs(sk_rf_acc - acc_rf) <= 0.05, "forest accuracy far from sklearn"
print("PASS: within 5% of sklearn's forest.")

sklearn RandomForest test acc = 0.9575 (ours: 0.9525)
PASS: within 5% of sklearn's forest.


## 7. Measuring $\rho$ and $\sigma^2$: bagging vs. random forest (notes §6.1-§6.2)

The notes' argument for random forests is about two numbers: $\sigma^2$, the
variance of one tree's prediction at a fixed input, and $\rho$, the
correlation between two trees' predictions at that input. Both are taken over
the randomness of the training set and of the bootstrap and feature draws.
To measure them we need many training sets, so this section uses a fresh
generator with one **dominant** feature: 6 standard-normal features, label
drawn from a logistic model with weights $(12, 3, 3, 0, 0, 0)$. With all
features available (`max_features = 6`, which is plain bagging), every tree
splits on feature 0 near the root, so the trees are similar and $\rho$ is
high.

For each `max_features` value $m$ from 1 to 6 we draw 20 training sets of 300
points, train 10 depth-8 trees on bootstrap samples of each, and record every
tree's predicted probability on 300 fixed test points. Per test point, the
code computes $\sigma^2$ (variance over all trees), the covariance between two
different trees trained on the same set ($\rho\sigma^2$), and their ratio
$\rho$, then averages over test points. It also reports the variance of the
10-tree average, which should equal $\rho\sigma^2 + (1-\rho)\sigma^2/10$
(the boxed formula), and the test accuracy of each 10-tree forest.

Expect $\rho$ to increase with $m$ and $\sigma^2$ to decrease with $m$, and
the floor $\rho\sigma^2$ (the part that more trees cannot remove) to be
lowest for small $m$.

In [8]:
D_DOM = 6
W_DOM = np.array([12.0, 3.0, 3.0, 0.0, 0.0, 0.0])


def make_dominant(n, rng):
    X = rng.normal(size=(n, D_DOM))
    p = 1.0 / (1.0 + np.exp(-(X @ W_DOM)))
    return X, (rng.random(n) < p).astype(int)


rng = np.random.default_rng(0)
X_fix, y_fix = make_dominant(300, rng)          # fixed test points
R, B, N_SET, DEPTH = 20, 10, 300, 8

results = {}
print(f"{'m':>2} {'sigma^2':>8} {'rho':>6} {'rho*sigma^2':>12} "
      f"{'Var(10-avg)':>12} {'formula':>8} {'test acc':>9}")
for m in range(1, D_DOM + 1):
    preds = np.zeros((R, B, len(X_fix)))    # [training set, tree, test point]
    accs = []
    for r in range(R):
        Xs, ys = make_dominant(N_SET, rng)
        for b in range(B):
            idx = rng.integers(0, N_SET, size=N_SET)
            tree = DecisionTree(max_depth=DEPTH, max_features=m, rng=rng)
            preds[r, b] = tree.fit(Xs[idx], ys[idx]).predict_proba(X_fix)
        accs.append(np.mean((preds[r].mean(axis=0) >= 0.5) == y_fix))

    mean = preds.mean(axis=(0, 1))                          # per test point
    sigma2 = ((preds - mean) ** 2).mean(axis=(0, 1))        # one-tree variance
    s = preds.sum(axis=1)                                   # (R, n_test)
    # mean of f_b * f_b' over pairs b != b' trained on the same set, minus mean^2
    cov = ((s ** 2 - (preds ** 2).sum(axis=1)) / (B * (B - 1))).mean(axis=0) - mean ** 2
    avg = preds.mean(axis=1)                                # 10-tree averages
    var_avg = ((avg - avg.mean(axis=0)) ** 2).mean()

    sig2, rho_sig2 = sigma2.mean(), cov.mean()
    rho = rho_sig2 / sig2
    formula = rho * sig2 + (1 - rho) * sig2 / B
    results[m] = dict(sigma2=sig2, rho=rho, floor=rho_sig2, acc=float(np.mean(accs)))
    label = "  (bagging)" if m == D_DOM else ""
    print(f"{m:>2} {sig2:8.4f} {rho:6.3f} {rho_sig2:12.4f} {var_avg:12.4f} "
          f"{formula:8.4f} {np.mean(accs):9.4f}{label}")

 m  sigma^2    rho  rho*sigma^2  Var(10-avg)  formula  test acc


 1   0.1302  0.037       0.0048       0.0173   0.0173    0.8892


 2   0.1013  0.055       0.0056       0.0151   0.0151    0.9043


 3   0.0800  0.079       0.0063       0.0136   0.0136    0.9128


 4   0.0708  0.112       0.0080       0.0142   0.0142    0.9090


 5   0.0645  0.131       0.0085       0.0141   0.0141    0.9125


 6   0.0609  0.167       0.0102       0.0153   0.0153    0.9137  (bagging)


## 8. What the table shows

The assertions below check the claims of notes §6.2 on these numbers:
restricting each split to fewer features lowers the correlation between trees
($\rho$ at $m = 2 \approx \sqrt{6}$ is about a third of bagging's), raises the
variance of a single tree ($\sigma^2$ goes up), and still lowers the floor
$\rho\sigma^2$, the part of the variance that remains however many trees are
averaged.

Two further points from the table. First, with only $B = 10$ trees the second
term $(1-\rho)\sigma^2/B$ is still larger than the floor, so the variance of
the 10-tree average is lowest at an intermediate $m$ (3 here), not at the
smallest $m$; the floor only dominates as $B$ grows. Second, at $m = 1$ many
splits are forced onto one of the three noise features, so each tree is both
noisier and less accurate on average, and the forest's accuracy is clearly
below the other settings. For $m \ge 3$ the accuracies are within about one
percentage point of each other. So $m$ has to be tuned.

In [9]:
bag, rf = results[D_DOM], results[2]
print(f"bagging (m={D_DOM}): rho = {bag['rho']:.3f}, sigma^2 = {bag['sigma2']:.4f}, "
      f"floor = {bag['floor']:.4f}")
print(f"random forest (m=2): rho = {rf['rho']:.3f}, sigma^2 = {rf['sigma2']:.4f}, "
      f"floor = {rf['floor']:.4f}")
assert rf["rho"] < bag["rho"], "feature subsampling did not lower rho"
assert rf["sigma2"] > bag["sigma2"], "feature subsampling did not raise sigma^2"
assert rf["floor"] < bag["floor"], "feature subsampling did not lower the floor"
best_m = max(results, key=lambda m: results[m]["acc"])
print(f"best test accuracy at m = {best_m}; m = 1 accuracy = {results[1]['acc']:.4f}")
assert results[1]["acc"] < results[best_m]["acc"]
print("PASS: lower rho, higher sigma^2, lower floor rho*sigma^2 with m = 2; m = 1 is too aggressive.")

bagging (m=6): rho = 0.167, sigma^2 = 0.0609, floor = 0.0102
random forest (m=2): rho = 0.055, sigma^2 = 0.1013, floor = 0.0056
best test accuracy at m = 6; m = 1 accuracy = 0.8892
PASS: lower rho, higher sigma^2, lower floor rho*sigma^2 with m = 2; m = 1 is too aggressive.
